# 금융분야 고객상담 데이터셋

In [5]:
# ----------------------------------------------------------------
# 디렉토리 내 데이터 개수 확인용
# ----------------------------------------------------------------
from pathlib import Path

file_path = "../dataset/25.금융분야 고객상담 데이터/3.개방데이터/2.데이터(NIA)/Training/02.라벨링데이터/"
# 파일명 예) 21-1_bk_01_000029_001

def count_json_files(base_path: str):
    base = Path(base_path)
    total = 0

    for i in range(1,10):
        sub_dir = base/f"{i:02d}" # i를 두 자리 숫자로 표현하는 형식 지정
        # file_path 내부에 01-09까지의 디렉토리가 존재하기 때문에 이들 각각을 접근하기 위함

        if not sub_dir.exists():
            print(f"[{sub_dir.name}] 디렉토리 없음") # .name : 경로의 마지막 부분 이름 반환
            continue

        # glob : 특정 패턴에 맞는 파일이나 폴더를 찾아주는 기능
        #        > sub_dir 디렉토리 내 .json으로 끝나는 모든 파일을 찾음
        # 이를 리스트로 변환 후 개수 반환
        count = len(list(sub_dir.glob("*.json")))
        total += count
        print(f"[{sub_dir.name}] {count}개")

    print(f"\n총 json 파일 개수 : {total}개")

if __name__ == "__main__":
    count_json_files(file_path)

[01] 14273개
[02] 4935개
[03] 17452개
[04] 6965개
[05] 7104개
[06] 11504개
[07] 3186개
[08] 12295개
[09] 2286개

총 json 파일 개수 : 80000개


In [7]:
# ----------------------------------------------------------------
# json 데이터 하나로 모아 df에 저장
# ----------------------------------------------------------------
import os
import json
import pandas as pd

# 데이터셋 경로
file_path = "../dataset/25.금융분야 고객상담 데이터/3.개방데이터/2.데이터(NIA)/Training/02.라벨링데이터/"

# 도메인, 중분류 코드 매핑
category_map = {
    "은행": {
        "01": "거래내역/잔액조회",
        "02": "중계요청/착오송금",
        "03": "자동이체조회",
        "04": "만기,연장/해지,수신",
        "05": "금융거래한도/비대면한도계좌",
        "06": "이자/연체금액",
        "07": "부수거래금리감면",
        "08": "대출문의(만기/연장/조회등)",
        "09": "환전문의",
    },

    "보험": {
        "01": "자동차보험상담",
        "02": "자동차사고접수",
        "03": "계약내용변경/해지",
        "04": "기타계약관련문의",
        "05": "보험금청구",
    },

    "증권": {
        "01": "HTS/MTS",
        "02": "계좌관리",
        "03": "신용거래/담보대출",
        "04": "자금이체/계좌제한",
        "05": "절세형금융상품",
        "06": "주식주문",
        "07": "증권계좌조회",
        "08": "해외주문",
    }
}

#파일명 분류코드 (도메인명으로 매핑하기 위함)
domain_map = {
    "bk" : "은행",
    "ins" : "보험",
    "sec" : "증권",
}

# json 내부 데이터 > df 형태로 변환
def flatten_json(data):
    """
    JSON 내부의 중첩된 dictionary를 펼쳐서
    DataFrame의 컬럼으로 사용할 수 있도록 변환
    """

    # --------------------------------
    # source
    # --------------------------------
    result = {}

    if "source" in data:
        source = data["source"]

        for key, value in source.items():
            result[f"source_{key}"] = value

    # --------------------------------
    # consulting
    # --------------------------------
    if "consulting" in data:
        consulting = data["consulting"]

        for key, value in consulting.items():
            result[f"consulting_{key}"] = value

    # --------------------------------
    # qa_data
    # --------------------------------
    if "qa_data" in data:

        # 하나의 JSON에 qa_data가 하나만 존재한다고 했으므로
        # 리스트의 첫 번째 데이터만 사용
        qa = data["qa_data"][0]

        for key, value in qa.items():

            # input은 dictionary이므로 한 번 더 펼침
            if key == "input" and isinstance(value, dict):

                for input_key, input_value in value.items():
                    result[f"qa_data_input_{input_key}"] = input_value

            else:
                result[f"qa_data_{key}"] = value

    return result

# 전체 json 파일 수집
all_data = []

# 01 ~ 08 디렉토리 순회
for folder_num in range(1, 10):

    folder_name = f"{folder_num:02d}"
    folder_path = os.path.join(file_path, folder_name)

    # 폴더가 존재하지 않는 경우 건너뜀
    if not os.path.exists(folder_path):
        print(f"[WARNING] 폴더 없음: {folder_path}")
        continue

    # 해당 폴더 내 JSON 파일 탐색
    for filename in os.listdir(folder_path):

        if not filename.lower().endswith(".json"):
            continue

        json_path = os.path.join(folder_path, filename)

        try:

            # -------------------------------
            # 파일명 분석
            # -------------------------------

            # 확장자 제거
            filename_without_ext = os.path.splitext(filename)[0]

            # 예:
            # 21-1_bk_01_000029_001
            parts = filename_without_ext.split("_")

            # 데이터분야별코드
            data_field_code = parts[0]

            # 분류코드
            domain_code = parts[1]

            # 중분류코드
            middle_code = parts[2]

            # 도메인명
            domain_name = domain_map.get(
                domain_code,
                f"Unknown({domain_code})"
            )

            # 중분류명
            middle_name = category_map.get(
                domain_name,
                {}
            ).get(
                middle_code,
                f"Unknown({middle_code})"
            )


            # -------------------------------
            # JSON 읽기
            # -------------------------------

            with open(
                json_path,
                "r",
                encoding="utf-8"
            ) as f:

                json_data = json.load(f)


            # JSON 내부 데이터 펼치기
            json_flattened = flatten_json(json_data)


            # -------------------------------
            # 메타데이터 + JSON 데이터 결합
            # -------------------------------

            row = {
                "파일명": filename,
                "데이터분야별코드": data_field_code,
                "분류코드": domain_code,
                "도메인명": domain_name,
                "중분류코드": middle_code,
                "중분류명": middle_name,
                "파일경로": json_path,
            }

            # JSON 내부 데이터 추가
            row.update(json_flattened)

            all_data.append(row)


        except Exception as e:

            print(f"[ERROR] {json_path}")
            print(f"       {e}")

# df 변환
df = pd.DataFrame(all_data)
df


,파일명,데이터분야별코드,분류코드,도메인명,중분류코드,중분류명,파일경로,source_source_institution,source_source_id,source_source_date,...,qa_data_consulting_situation,qa_data_qa_topic,qa_data_consulting_purpose,qa_data_core_financial_terms,qa_data_input_length,qa_data_instruction,qa_data_input_question,qa_data_input_answer,qa_data_input_follow_up_question,qa_data_output
0,21-1_bk_01_000029_001.json,21-1,bk,은행,01,거래내역/잔액조회,../dataset/25.금융분야 고객상담 데이터/3.개방데이터/2.데이터(NIA)...,하나은행,21-1_bk_01_000029,202506,...,민원응대,대출문의(만기/연장/조회등),입금 차액 확인,계약금,212,입금 차액과 자동 정정 시점에 대한 정보를 추출하고 정리하시오.,입주 계약과 관련된 대출 계약금 관련해서 제가 대부계 계약금을 보내드렸는데 제 통...,시스템 상에서 입금 금액과 실제 통장에 반영된 금액 사이에 일시적인 연동 오류가 발...,"그렇다면 오류가 자동으로 정정되는 정확한 시점은 언제인지, 그리고 정정이 안 될 경...",입금 금액과 통장에 반영된 금액 사이에 일시적인 시스템 연동 오류가 발생할 수 있으...
1,21-1_bk_01_000029_002.json,21-1,bk,은행,01,거래내역/잔액조회,../dataset/25.금융분야 고객상담 데이터/3.개방데이터/2.데이터(NIA)...,하나은행,21-1_bk_01_000029,202506,...,민원응대,대출문의(만기/연장/조회등),할인혜택 문의,NaN,217,할인혜택 적용 기준과 재신청 절차를 안내하시오.,제가 입금받은 금액에 대해 할인 혜택을 받으려면 통장에 정확히 입금된 금액을 유지해...,"말씀하신 할인 혜택은 실제 입금 확인이 된 금액을 기준으로 적용되기 때문에, 차이가...",그렇다면 정정이 완료된 뒤에 할인 혜택을 재신청하는 절차는 어떻게 진행해야 하나요?...,"정정이 완료되면 문자와 이메일로 안내해 드리며, 안내에 따라 온라인 뱅킹이나 모바일..."
2,21-1_bk_01_000075_002.json,21-1,bk,은행,01,거래내역/잔액조회,../dataset/25.금융분야 고객상담 데이터/3.개방데이터/2.데이터(NIA)...,하나은행,21-1_bk_01_000075,202505,...,일반문의,거래내역/잔액조회,타행이체및 계좌 일시정지 문의,계좌이체,248,타행 이체 절차와 계좌 일시 정지 및 출금 내역 확인 방법을 안내하시오.,금액을 다른 계좌로 이체하고 싶을 때는 어떤 절차를 밟아야 하나요? 그리고 현재 계...,다른 계좌로 이체하시려면 먼저 모바일 뱅킹 또는 인터넷 뱅킹에 로그인 후 이체 메뉴...,모바일 뱅킹 앱에서 이번 출금 내역을 확인하고 싶을 때는 어떤 메뉴를 눌러야 하나요?,다른 계좌로 이체하실 때는 모바일 뱅킹이나 인터넷 뱅킹에 로그인하신 후 이체 메뉴에...
3,21-1_bk_01_000085_001.json,21-1,bk,은행,01,거래내역/잔액조회,../dataset/25.금융분야 고객상담 데이터/3.개방데이터/2.데이터(NIA)...,하나은행,21-1_bk_01_000085,202506,...,일반문의,환전문의,카드발급 문의,환전,312,"외화와 원화 계좌간 이체 방법, 인증번호 재발송 방법, 인터넷뱅킹 외의 인증 수단을...",새로운 카드를 만들고 싶어서 전화드렸습니다. 보유하고 있는 계좌는 외화 계좌라서 원...,"외화 계좌에서 원화 계좌로 자금을 이동하고, 그 후 카드 발급을 위한 최소 예치금 ...","인증 번호를 받지 못했는데, 인증 번호를 재발송 받으려면 어떻게 해야 하나요? 인터...",외화 계좌에서 원화 계좌로 입금하시려면 외화 계좌에 로그인 후 환전 송금 메뉴에서 ...
4,21-1_bk_01_000085_002.json,21-1,bk,은행,01,거래내역/잔액조회,../dataset/25.금융분야 고객상담 데이터/3.개방데이터/2.데이터(NIA)...,하나은행,21-1_bk_01_000085,202506,...,일반문의,거래내역/잔액조회,입금내역 확인,입금,336,입금내역 확인 방과 카드 발급시 필요서류를 안내하시오.,입금내역을 확인하고 싶을 때 제가 직접 전화로 요청하면 바로 확인이 가능한가요? 아...,전화로 입금내역을 요청하시면 상담원이 실시간으로 확인 후 바로 안내해 드립니다 다만...,카드 발급 절차 중에 제가 직접 방문해야 하는 서류가 있나요? 예를 들어 신분증 사...,"전화로 입금 내역을 요청하시면 영업시간에는 상담원이 실시간으로 안내해 드리며, 영업..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79995,21-1_bk_09_029027_002.json,21-1,bk,은행,09,환전문의,../dataset/25.금융분야 고객상담 데이터/3.개방데이터/2.데이터(NIA)...,하나은행,21-1_bk_09_029027,202505,...,일반문의,환전문의,환전 절차,환전,203,전화 신청을 통한 환전 절차와 자동 이체 조건을 설명하시오.,환전 신청을 전화로만 진행할 수 있는지 궁금합니다. 전화 신청 절차는 어떻게 되는지...,"네 고객님, 환전 신청은 전화로도 가능합니다. 전화 신청 시 본인 확인 절차가 필요...","환전 신청을 전화로만 진행할 수 있나요, 자동 이체 조건은 어떻게 되나요?","고객님께서는 환전 신청을 전화로도 하실 수 있으며, 이 경우 본인 확인을 위해 계좌..."
79996,21-1_bk_09_029027_003.json,21-1,bk,은행,09,환전문의,../dataset/25.금융분야 고객상담 데이터/3.개방데이터/2.데이터(NIA)...,하나은행,21-1_bk_09_029027,202505,...,일반문의,환전문의,수수료 및 환율 적용,환전,246,환전 시 수수료 정책과 환율 적용 기준을 설명하시오.,"환전 시 적용되는 수수료와 환율 기준이 궁금합니다. 수수료는 몇 퍼센트인지, 환율은...","네 고객님 환전 시 수수료는 기본적으로 0.5%이며, 적용 환율은 실시간 기준으로 ...",환전 시 수수료와 환율은 어떻게 적용되나요?,고객님 환전 수수료는 기본 0.5%이며 환율은 실시간으로 적용되어 거래 시점에 따라...
79997,21-1_bk_09_029029_001.json,21-1,bk,은행,09,환전문의,../dataset/25.금융분야 고객상담 데이터/3.개방데이터/2.데이터(NIA)...,하나은행,21-1_bk_09_029029,202505,...,일반문의,환전문의,외화 환전 가능 여부,환전,257,◆◆ 금융 센터의 외화 환전 가능 여부와 환율·수수료·방문 절차를 설명하시오.,◆◆은행 ◆◆ 금융 센터에서 외화 환전이 가능한지 확인하고 싶습니다. 해당 지점의 ...,네 고객님 ◆◆ 금융 센터에서는 현재 외화 보유가 없어 환전 서비스 제공이 어려운 ...,"◆◆ 본사에 문의했을 때 추가 확인이 필요하다고 했는데, 다시 확인해 주실 수 있나요?",고객님께서는 현재 해당 금융 센터에서 외화 보유가 없어 환전 서비스가 어려운 점 양...
79998,21-1_bk_09_029029_002.json,21-1,bk,은행,09,환전문의,../dataset/25.금융분야 고객상담 데이터/3.개방데이터/2.데이터(NIA)...,하나은행,21-1_bk_09_029029,202505,...,일반문의,환전문의,외화 보유 현황,환전수수료,219,"지점별 외화 보유 현황과 권종 차이, 환전 절차를 설명하시오.","각 지점마다 외화 보유 현황이 다른지 궁금합니다. ◆◆지점에는 외화가 있는지, 권종...",네 고객님 각 지점마다 외화 보유 현황과 권종이 다를 수 있습니다. ◆◆지점에는 일...,"◆◆지점에 외화가 있다면 바로 환전할 수 있나요, 권종은 어떻게 확인하

In [8]:
# ----------------------------------------------------------------
# 추출한 열 확인용
# ----------------------------------------------------------------
df.columns

Index(['파일명', '데이터분야별코드', '분류코드', '도메인명', '중분류코드', '중분류명', '파일경로',
       'source_source_institution', 'source_source_id', 'source_source_date',
       'source_client_gender', 'source_client_age', 'source_consulting_client',
       'source_consulting_client_type', 'source_source_length',
       'source_consulting_content', 'consulting_consulting_category',
       'consulting_consulting_topic', 'consulting_consulting_summary',
       'qa_data_qa_id', 'qa_data_task_category',
       'qa_data_consulting_situation', 'qa_data_qa_topic',
       'qa_data_consulting_purpose', 'qa_data_core_financial_terms',
       'qa_data_input_length', 'qa_data_instruction', 'qa_data_input_question',
       'qa_data_input_answer', 'qa_data_input_follow_up_question',
       'qa_data_output'],
      dtype='str')

In [10]:
# ----------------------------------------------------------------
# 생성한 df 엑셀 파일로 저장
# ----------------------------------------------------------------
df.to_excel("output.xlsx", index=False)

In [11]:
# ----------------------------------------------------------------
# 저장한 데이터셋 엑셀파일 불러오기
# ----------------------------------------------------------------
import pandas as pd

file_path = "../dataset/output.xlsx"

df = pd.read_excel(file_path)
df

BadZipFile: File is not a zip file

In [ ]:
'''
Index(['파일명', '데이터분야별코드', '분류코드', '도메인명', '중분류코드', '중분류명', '파일경로',
       'source_source_institution', 'source_source_id', 'source_source_date',
       'source_client_gender', 'source_client_age', 'source_consulting_client',
       'source_consulting_client_type', 'source_source_length',
       'source_consulting_content', 'consulting_consulting_category',
       'consulting_consulting_topic', 'consulting_consulting_summary',
       'qa_data_qa_id', 'qa_data_task_category',
       'qa_data_consulting_situation', 'qa_data_qa_topic',
       'qa_data_consulting_purpose', 'qa_data_core_financial_terms',
       'qa_data_input_length', 'qa_data_instruction', 'qa_data_input_question',
       'qa_data_input_answer', 'qa_data_input_follow_up_question',
       'qa_data_output'],
      dtype='str')
'''